# 🛠️ Practice Project Template: End-to-End Customer Churn Prediction
## Real-World Tabular Classification, Pipeline Engineering & Evaluation

Customer churn prediction is one of the most vital machine learning applications in telecommunications, SaaS, banking, and e-commerce.

---

### 📋 Project Roadmap:
1. **Exploratory Data Analysis (EDA)**: Inspect churn distributions, missing values, and feature correlations.
2. **Feature Engineering & Preprocessing**:
   - Numerical: Imputation & Feature Scaling (`StandardScaler`)
   - Categorical: One-Hot Encoding (`OneHotEncoder`)
   - Combining cleanly inside Scikit-learn's `ColumnTransformer`
3. **Pipeline Construction**: Leak-free training with `Pipeline`.
4. **Model Comparison**:
   - Logistic Regression (Baseline)
   - Random Forest Classifier
   - Gradient Boosting Classifier
5. **Business Diagnostics**:
   - Confusion Matrix & PR-AUC analysis
   - Optimal Decision Threshold Selection for revenue retention!


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score

# Load Dataset
df = pd.read_csv("telecom_churn_data.csv")
print(f"Dataset shape: {df.shape}")
df.head()


---
## Step 1: Exploratory Data Analysis & Target Split


In [ ]:
# Inspect Churn Rate
churn_counts = df['churn'].value_counts(normalize=True)
print(f"Retained (0): {churn_counts[0]*100:.1f}%, Churned (1): {churn_counts[1]*100:.1f}%")

# Feature definition
X = df.drop(columns=['customer_id', 'churn'])
y = df['churn']

num_cols = ['tenure_months', 'monthly_charges', 'total_charges']
cat_cols = ['contract_type', 'internet_service', 'tech_support']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
print(f"Train set: {X_train.shape[0]} rows, Test set: {X_test.shape[0]} rows")


---
## Step 2: [YOUR TURN] Build the Preprocessing `ColumnTransformer`
**Instructions:**
- Create `num_pipeline` with `SimpleImputer(strategy='median')` and `StandardScaler()`.
- Create `cat_pipeline` with `SimpleImputer(strategy='most_frequent')` and `OneHotEncoder(handle_unknown='ignore')`.
- Assemble them into a `preprocessor = ColumnTransformer(...)`.


In [ ]:
### YOUR CODE HERE: Construct Preprocessing Pipelines ###
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

print("Preprocessor assembled successfully!")


---
## Step 3: [YOUR TURN] Model Training & Benchmarking
**Instructions:**
- Test 3 pipelines: Logistic Regression, Random Forest, and Gradient Boosting.
- Record Accuracy, ROC-AUC, and PR-AUC on the test set.


In [ ]:
candidate_models = {
    "Logistic Regression": LogisticRegression(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, learning_rate=0.08, max_depth=4, random_state=42)
}

benchmark_results = []

for name, clf in candidate_models.items():
    pipe = Pipeline([
        ('prep', preprocessor),
        ('model', clf)
    ])
    
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]
    
    roc = roc_auc_score(y_test, y_prob)
    pr_auc = average_precision_score(y_test, y_prob)
    
    benchmark_results.append({
        "Model": name,
        "ROC-AUC": round(roc, 4),
        "PR-AUC": round(pr_auc, 4)
    })

res_df = pd.DataFrame(benchmark_results)
print("=== MODEL BENCHMARK RESULTS ===")
print(res_df.to_string(index=False))


---
## Step 4: Decision Threshold Tuning
In customer churn, reaching out with a discount retention offer to an at-risk customer is cheaper than losing them permanently. Let's tune the threshold to maximize Recall while preserving reasonable Precision!


In [ ]:
best_pipe = Pipeline([
    ('prep', preprocessor),
    ('model', GradientBoostingClassifier(n_estimators=100, random_state=42))
])
best_pipe.fit(X_train, y_train)
test_probs = best_pipe.predict_proba(X_test)[:, 1]

thresholds = [0.2, 0.3, 0.4, 0.5, 0.6]
for th in thresholds:
    preds_th = (test_probs >= th).astype(int)
    cm = confusion_matrix(y_test, preds_th)
    recall = cm[1, 1] / (cm[1, 1] + cm[1, 0])
    precision = cm[1, 1] / (cm[1, 1] + cm[0, 1]) if (cm[1, 1] + cm[0, 1]) > 0 else 0
    print(f"Threshold: {th:.2f} | Recall: {recall*100:.1f}% | Precision: {precision*100:.1f}% | Caught Churners: {cm[1, 1]}/{cm[1, 1] + cm[1, 0]}")


---
## 🏆 Practice Challenge Extensions:
1. **Add Optuna Tuning**: Optimize `learning_rate`, `n_estimators`, and `max_depth` using Optuna.
2. **Feature Importance**: Extract feature names from the `OneHotEncoder` and plot the top 10 most predictive features!
